# Stage 21A: visible-prefix candidate router

Stage 19/20の3係数残差を終了し、well自身の既知prefix内backtestで候補を選びます。A100/A130/A160 PF、top-PF blend、public OOF、U多項式を2つのinternal cutsで評価し、選ばれた候補を固定25%・12 ft上限でbaseへ混ぜます。Stage 20A/Bの全discovery wellsを除外し、Kaggle提出は作りません。CPUランタイムを使用してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定artifactと除外well

Stage 17Aのtarget-safe cut定義と、Stage 7のwell-isolated`base_oof.parquet`を使います。Stage 20A/Bで使ったwellは両方除外します。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage20a_run=artifact_dir/'stage20a_top_pf_alignment_full_v001'
stage20b_run=artifact_dir/'stage20b_disjoint_confirmation_full_v001'
for path,name in [(stage16b_run/'well_assignments.parquet','stage16b'),(stage17a_run/'cut_report.parquet','stage17a'),(public_oof_run/'base_oof.parquet','public_oof'),(stage20a_run/'cut_features.parquet','stage20a'),(stage20b_run/'cut_features.parquet','stage20b')]:
    assert path.is_file(),f'{name}: {path}'
print(stage16b_run,stage17a_run,public_oof_run,stage20a_run,stage20b_run,sep='\n')


## Prefix backtest router

各outer cutでouter PF 3本とinternal PF 6本を計算します。Stage 20より1 cutあたりの計算は多いですが、particles/seeds/stepsをscreen用に縮小しています。10 cutsごとに進捗を表示します。


In [ ]:
RUN_ID='stage21a_prefix_router_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved)
    shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    run_checked(['uv','run','rogii-prefix-router','--config','configs/experiment/stage21a_prefix_router.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--exclude-run',str(stage20a_run),'--exclude-run',str(stage20b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage21a_complete','promoted_to_stage21b','sample_cuts','sample_wells','excluded_discovery_wells','discovery_well_overlap','candidate_count','rank_correlation','top1_oracle_match_fraction','base_rmse','guarded_router_rmse','guarded_router_delta','raw_router_rmse','oracle_rmse','oracle_delta','well_p90_delta','bootstrap_95pct','gates','next_step']}


In [ ]:
import pandas as pd
candidates=pd.read_parquet(run_dir/'candidate_report.parquet')
candidates.groupby('candidate').agg(cuts=('cut_id','nunique'),inner_rmse=('inner_rmse','mean'),outer_rmse=('outer_rmse','mean'),selected=('selected','sum')).sort_values('outer_rmse')

最後の辞書とcandidate表を共有してください。全gate通過時だけ、別wellかつ高PF解像度のStage 21Bへ進みます。このscreenから直接router modelやsubmissionを作りません。
